# Dataquest 2023 Preliminary: XGBoost with time and season features, 5-fold CV (submission score 0.79679)

In [ ]:
!pip install -q catboost pycaret autoviz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import catboost as cb
import xgboost as xgb
import lightgbm as lgb

import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_DIR = "../data"
SUBMISSION_DIR = "../submissions"

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df.head()

,datetime,datetime_iso,time-zone,temp,visibility,d_point,feels,min_temp,max_temp,prssr,sea_level,grnd_level,hum,wind_spd,wind_deg,rain_1h,rain_3h,snow_1h,snow_3h,clouds
0,283996800,1979-01-01 00:00:00+00:00,28800,24.75 Celcius,NaN,23.89 C,25.76 C,24.28,25.22°C,1012,undetermined,NaN,95,0.82,320.0 °,zero,0,NaN,NaN,100
1,284000400,1979-01-01 01:00:00+00:00,28800,24.58 C,NaN,23.73 C,25.57 C,23.99 C,25.26 C,1012,NaN,NaN,95,0.96 m/s,338.0°,0,0,0,0,100
2,284004000,1979-01-01 02:00:00+00:00,28800,26.6 Celcius,unidentified,24.06 C,26.6 C,26.1 C,27.39,1012,NaN,undetermined,86,1.22 m/s,339.0°,0,volume:zero,NaN,NaN,99
3,284007600,1979-01-01 03:00:00+00:00,28800,27.31 Celcius,NaN,24.37 C,30.9 C,26.59,28.36 C,1012,NaN,undetermined,84,1.08 m/s,342,0.13,nol,0,NaN,94
4,284011200,1979-01-01 04:00:00+00:00,28800,27.41,NaN,25.05 C,31.54 C,26.58 C,28.31 °C,1011,NaN,undetermined,87,0.86,336.0°,0.34,nol,NaN,0,100


In [ ]:
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

In [ ]:
ori_test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

In [ ]:
test_df.head()

,datetime,datetime_iso,time-zone,temp,visibility,d_point,feels,min_temp,max_temp,prssr,sea_level,grnd_level,hum,wind_spd,wind_deg,rain_3h,snow_1h,snow_3h,clouds
0,1514764800,2018-01-01 00:00:00+00:00,28800,26.59 °C,NaN,23.66,26.59,26.02,27.16,1009,NaN,undetermined,84,1.45 m/s,355,0,NaN,no_snow,97
1,1514768400,2018-01-01 01:00:00+00:00,28800,26.51 C,NaN,24.92,26.51 °C,26.06,28.04,1009,NaN,undetermined,91,1.67 m/s,351,0mm,no-snow,0 milimeter,95
2,1514772000,2018-01-01 02:00:00+00:00,28800,28.68 C,NaN,25.71,34.68,28.03 C,29.3 C,1009.0 hPa.,NaN,NaN,84,1.72 m/s,345.0°,0 mm,volume:zero,0,90
3,1514775600,2018-01-01 03:00:00+00:00,28800,28.84 C,NaN,25.25,34.51,28.52,29.08 °C,1008,NaN,NaN,81,1.49,339.0°,0 milimeter,0,NaN,91
4,1514779200,2018-01-01 04:00:00+00:00,28800,29.75 Celcius,,24.62,35.38,29.31 C,30.57,1007,undetermined,undetermined,74,1.39 m/s,339.0°,NaN,0,volume:0,96


In [ ]:
print(train_df['clouds'].apply(
    lambda x: float(re.search(r'\d+(\.\d+)?', x).group(0)) if re.search(r'\d+(\.\d+)?', x) else None).isnull().sum())
train_df['clouds'].apply(
    lambda x: float(re.search(r'\d+(\.\d+)?', x).group(0)) if re.search(r'\d+(\.\d+)?', x) else None)

0


0         100.0
1         100.0
2          99.0
3          94.0
4         100.0
          ...  
341875     99.0
341876    100.0
341877     97.0
341878     98.0
341879     99.0
Name: clouds, Length: 341880, dtype: float64

In [ ]:
def clean_cols(df, is_train=True):
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s')

    # function to clean unit
    def clean_unit(x):
        temp_pattern = r'-?\d+(\.\d+)?'
        regex = re.search(temp_pattern, str(x))
        return float(regex.group(0)) if regex else np.nan

    # clean unit
    df['temp'] = df['temp'].apply(clean_unit)
    df['d_point'] = df['d_point'].apply(clean_unit)
    df['feels'] = df['feels'].apply(clean_unit)
    df['min_temp'] = df['min_temp'].apply(clean_unit)
    df['max_temp'] = df['max_temp'].apply(clean_unit)
    df['prssr'] = df['prssr'].apply(clean_unit)
    df['hum'] = df['hum'].apply(clean_unit) / 100
    df['wind_spd'] = df['wind_spd'].apply(clean_unit)
    df['wind_deg'] = df['wind_deg'].apply(clean_unit)
    df['clouds'] = df['clouds'].apply(clean_unit) / 100

    df.loc[df['hum'] > 1, 'hum'] = 1

    # clean rain_1h
    if is_train:
        df.loc[df['rain_1h'] == 'zero', 'rain_1h'] = 0
        df.loc[df['rain_1h'] == ' ', 'rain_1h'] = np.nan
        df.dropna(subset=['rain_1h'], inplace=True)
        df['rain_1h'] = df['rain_1h'].apply(clean_unit)
        df.loc[df['rain_1h'] < 0, 'rain_1h'] = 0

    # drop useless columns
    df.drop(['datetime_iso',
             'time-zone',
             'visibility',
             'sea_level',
             'grnd_level',
             'rain_3h',
             'snow_1h',
             'snow_3h'],
            axis=1, inplace=True)

    return df

In [ ]:
train_df = clean_cols(train_df)
train_df.head()

,datetime,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,rain_1h,clouds
0,1979-01-01 00:00:00,24.75,23.89,25.76,24.28,25.22,1012.0,0.95,0.82,320.0,0.00,1.00
1,1979-01-01 01:00:00,24.58,23.73,25.57,23.99,25.26,1012.0,0.95,0.96,338.0,0.00,1.00
2,1979-01-01 02:00:00,26.60,24.06,26.60,26.10,27.39,1012.0,0.86,1.22,339.0,0.00,0.99
3,1979-01-01 03:00:00,27.31,24.37,30.90,26.59,28.36,1012.0,0.84,1.08,342.0,0.13,0.94
4,1979-01-01 04:00:00,27.41,25.05,31.54,26.58,28.31,1011.0,0.87,0.86,336.0,0.34,1.00


In [ ]:
train_df['year'] = train_df['datetime'].dt.year
train_df['month'] = train_df['datetime'].dt.month
train_df['week'] = train_df['datetime'].dt.week
train_df['day'] = train_df['datetime'].dt.day
train_df['hour'] = train_df['datetime'].dt.hour
train_df['dayofweek'] = train_df['datetime'].dt.dayofweek

In [ ]:
test_df = clean_cols(test_df, is_train=False)
test_df.head()

,datetime,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,clouds
0,2018-01-01 00:00:00,26.59,23.66,26.59,26.02,27.16,1009.0,0.84,1.45,355.0,0.97
1,2018-01-01 01:00:00,26.51,24.92,26.51,26.06,28.04,1009.0,0.91,1.67,351.0,0.95
2,2018-01-01 02:00:00,28.68,25.71,34.68,28.03,29.30,1009.0,0.84,1.72,345.0,0.90
3,2018-01-01 03:00:00,28.84,25.25,34.51,28.52,29.08,1008.0,0.81,1.49,339.0,0.91
4,2018-01-01 04:00:00,29.75,24.62,35.38,29.31,30.57,1007.0,0.74,1.39,339.0,0.96


In [ ]:
test_df['year'] = test_df['datetime'].dt.year
test_df['month'] = test_df['datetime'].dt.month
test_df['week'] = test_df['datetime'].dt.week
test_df['day'] = test_df['datetime'].dt.day
test_df['hour'] = test_df['datetime'].dt.hour
test_df['dayofweek'] = test_df['datetime'].dt.dayofweek

In [ ]:
test_df.drop(columns='datetime', inplace=True)

In [ ]:
train_df.drop(columns='datetime', inplace=True)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 320590 entries, 0 to 341879
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   temp       320590 non-null  float64
 1   d_point    320590 non-null  float64
 2   feels      320590 non-null  float64
 3   min_temp   320590 non-null  float64
 4   max_temp   320590 non-null  float64
 5   prssr      320590 non-null  float64
 6   hum        320590 non-null  float64
 7   wind_spd   320590 non-null  float64
 8   wind_deg   320590 non-null  float64
 9   rain_1h    320590 non-null  float64
 10  clouds     320590 non-null  float64
 11  year       320590 non-null  int64  
 12  month      320590 non-null  int64  
 13  week       320590 non-null  int64  
 14  day        320590 non-null  int64  
 15  hour       320590 non-null  int64  
 16  dayofweek  320590 non-null  int64  
dtypes: float64(11), int64(6)
memory usage: 44.0 MB


In [ ]:
import pandas as pd

def create_features(df):
    # Temperature Range
    df['temp_range'] = df['max_temp'] - df['min_temp']

    # Weekend Indicator
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    # Daytime Indicator
    df['is_daytime'] = ((df['hour'] >= 6) & (df['hour'] <= 18)).astype(int)
    # Season Indicator
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        else:
            return 'Fall'
    df['season'] = df['month'].apply(get_season)
    import pandas as pd

    # Encode 'season' column as one-hot encoding
    season_encoded = pd.get_dummies(df['season'], prefix='season')

    # Concatenate the one-hot encoded columns with the original DataFrame
    df_encoded = pd.concat([df, season_encoded], axis=1)

    # Drop the original 'season' column
    df_encoded.drop('season', axis=1, inplace=True)


    return df_encoded

# Usage:
# Assuming you have a DataFrame called train_df
train_df = create_features(train_df)
test_df = create_features(test_df)

In [ ]:
train_df.head()

,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,rain_1h,clouds,year,month,week,day,hour,dayofweek,temp_range,is_weekend,is_daytime,season_Fall,season_Spring,season_Summer,season_Winter
0,24.75,23.89,25.76,24.28,25.22,1012.0,0.95,0.82,320.0,0.00,1.00,1979,1,1,1,0,0,0.94,0,0,0,0,0,1
1,24.58,23.73,25.57,23.99,25.26,1012.0,0.95,0.96,338.0,0.00,1.00,1979,1,1,1,1,0,1.27,0,0,0,0,0,1
2,26.60,24.06,26.60,26.10,27.39,1012.0,0.86,1.22,339.0,0.00,0.99,1979,1,1,1,2,0,1.29,0,0,0,0,0,1
3,27.31,24.37,30.90,26.59,28.36,1012.0,0.84,1.08,342.0,0.13,0.94,1979,1,1,1,3,0,1.77,0,0,0,0,0,1
4,27.41,25.05,31.54,26.58,28.31,1011.0,0.87,0.86,336.0,0.34,1.00,1979,1,1,1,4,0,1.73,0,0,0,0,0,1


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

# Assuming you have a train_df and test_df with your data and target variable

# Define your features (X) and target variable (y)
X_train = train_df.drop(columns=['rain_1h'])
y_train = train_df['rain_1h']
X_test = test_df  # Features for prediction

# Initialize the XGBoost regressor
xgb_model = xgb.XGBRegressor(n_estimators=30,objective='reg:squarederror', random_state=42)  # You can tune hyperparameters as needed

# Evaluate RMSE using cross-validation
rmse_scores = np.sqrt(-cross_val_score(xgb_model, X_train, y_train, scoring="neg_mean_squared_error", cv=5))

# Fit the model on the entire training dataset
xgb_model.fit(X_train, y_train)

# Predict on the test dataset
y_pred = xgb_model.predict(X_test)

# Optionally, you can save the predictions to a DataFrame
predictions_df = pd.DataFrame({'Predicted_Target': y_pred})

# Print the RMSE scores from cross-validation
print("Cross-Validation RMSE Scores:", rmse_scores)
print("Mean RMSE:", np.mean(rmse_scores))

Cross-Validation RMSE Scores: [0.78139112 0.78657021 0.80817874 0.85413189 0.90571889]
Mean RMSE: 0.8271981692113173


In [ ]:
predictions_df.head()

,Predicted_Target
0,-0.085987
1,0.007652
2,0.113742
3,0.194365
4,0.184429


In [ ]:
test_df

,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,clouds,year,month,week,day,hour,dayofweek,temp_range,is_weekend,is_daytime,season_Fall,season_Spring,season_Summer,season_Winter
0,26.59,23.66,26.59,26.02,27.16,1009.0,0.84,1.45,355.0,0.97,2018,1,1,1,0,0,1.14,0,0,0,0,0,1
1,26.51,24.92,26.51,26.06,28.04,1009.0,0.91,1.67,351.0,0.95,2018,1,1,1,1,0,1.98,0,0,0,0,0,1
2,28.68,25.71,34.68,28.03,29.30,1009.0,0.84,1.72,345.0,0.90,2018,1,1,1,2,0,1.27,0,0,0,0,0,1
3,28.84,25.25,34.51,28.52,29.08,1008.0,0.81,1.49,339.0,0.91,2018,1,1,1,3,0,0.56,0,0,0,0,0,1
4,29.75,24.62,35.38,29.31,30.57,1007.0,0.74,1.39,339.0,0.96,2018,1,1,1,4,0,1.26,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49363,24.37,23.34,25.32,22.70,28.20,1011.0,0.94,1.57,239.0,0.84,2023,8,33,19,19,5,5.50,1,0,0,0,1,0
49364,23.87,23.02,24.79,21.91,28.01,1011.0,0.95,1.53,235.0,0.70,2023,8,33,19,20,5,6.10,1,0,0,0,1,0
49365,23.87,23.02,24.79,21.91,28.01,1011.0,0.95,1.53,235.0,0.70,2023,8,33,19,21,5,6.10,1,0,0,0,1,0
49366,23.87,23.02,24.79,21.91,28.01,1011.0,0.95,1.53,235.0,0.70,2023,8,33,19,22,5,6.10,1,0,0,0,1,0


In [ ]:
predictions_df['datetime_iso'] = ori_test_df['datetime_iso']

In [ ]:
predictions_df.rename(columns={'Predicted_Target': 'rain_1h'}, inplace=True)

In [ ]:
predictions_df.head()

,rain_1h,datetime_iso
0,0.000000,2018-01-01 00:00:00+00:00
1,0.007652,2018-01-01 01:00:00+00:00
2,0.113742,2018-01-01 02:00:00+00:00
3,0.194365,2018-01-01 03:00:00+00:00
4,0.184429,2018-01-01 04:00:00+00:00


In [ ]:
predictions_df.loc[predictions_df['rain_1h'] < 0, 'rain_1h'] = 0

In [ ]:
# Optionally, you can save the predictions to a CSV file
predictions_df.to_csv(f"{SUBMISSION_DIR}/prediction3.csv", index=False)